In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EmployeePipeline") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("✅ Spark Started")


✅ Spark Started


In [2]:
df = spark.read.csv("employees_raw.csv", header=True, inferSchema=True)

df.show(5)


+-----------------+---------------+-----------+--------------------+--------------------+---------------+----------+-------+----------+--------------------+----+-----+--------+----------+------+
|      employee_id|     first_name|  last_name|               email|           hire_date|      job_title|department| salary|manager_id|             address|city|state|zip_code|birth_date|status|
+-----------------+---------------+-----------+--------------------+--------------------+---------------+----------+-------+----------+--------------------+----+-----+--------+----------+------+
|                0|       Michelle|     Miller|   preed@example.com| 2021-07-17 00:00:00|Patent attorney| analytics|$34,634|      1474|7699 Kenneth Lodg...|NULL| NULL|    NULL|      NULL|  NULL|
|South Stephenfort|      LA 00953"|Bryantshire|                  MH|+40146-01-01 00:0...|     1977-03-16|    Active|   NULL|      NULL|                NULL|NULL| NULL|    NULL|      NULL|  NULL|
|                1|      

In [3]:
df = df.dropDuplicates(["employee_id"])

In [4]:
df = df.dropna(subset=["employee_id", "email"])

In [5]:
from pyspark.sql.functions import *
df = df.withColumn("first_name", initcap(col("first_name"))) \
       .withColumn("last_name", initcap(col("last_name")))

df = df.withColumn("email", lower(col("email")))
df = df.filter(col("email").contains("@"))

df = df.withColumn("salary",
    regexp_replace("salary", "[$,]", "").cast("double")
)
df = df.withColumn("age",
    floor(datediff(current_date(), col("birth_date")) / 365)
)
df = df.withColumn("tenure_years",
    round(datediff(current_date(), col("hire_date")) / 365, 1)
)
df = df.withColumn("salary_band",
    when(col("salary") < 50000, "Junior")
    .when((col("salary") >= 50000) & (col("salary") <= 80000), "Mid")
    .otherwise("Senior")
)

In [6]:
df = df.withColumn("full_name",
    concat(col("first_name"), lit(" "), col("last_name"))
)
df = df.withColumn("email_domain",
    split(col("email"), "@").getItem(1)
)
df.show(10)
df.printSchema()

+-----------+-----------+---------+--------------------+-------------------+--------------------+----------+-------+----------+--------------------+----+-----+--------+----------+------+----+------------+-----------+------------------+------------+
|employee_id| first_name|last_name|               email|          hire_date|           job_title|department| salary|manager_id|             address|city|state|zip_code|birth_date|status| age|tenure_years|salary_band|         full_name|email_domain|
+-----------+-----------+---------+--------------------+-------------------+--------------------+----------+-------+----------+--------------------+----+-----+--------+----------+------+----+------------+-----------+------------------+------------+
|          0|   Michelle|   Miller|   preed@example.com|2021-07-17 00:00:00|     Patent attorney| analytics|34634.0|      1474|7699 Kenneth Lodg...|NULL| NULL|    NULL|      NULL|  NULL|NULL|         4.7|     Junior|   Michelle Miller| example.com|
|   

In [8]:
df.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/postgres") \
    .option("dbtable", "employees_clean") \
    .option("user", "postgres") \
    .option("password", "Sakshi@2803") \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()

print("✅ Data Loaded Successfully")

✅ Data Loaded Successfully


In [9]:
spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/postgres") \
    .option("dbtable", "employees_clean") \
    .option("user", "postgres") \
    .option("password", "Sakshi@2803") \
    .option("driver", "org.postgresql.Driver") \
    .load().show(5)

+-----------+----------+---------+--------------------+-------------------+--------------------+----------+-------+----------+--------------------+----+-----+--------+----------+------+----+------------+-----------+---------------+------------+
|employee_id|first_name|last_name|               email|          hire_date|           job_title|department| salary|manager_id|             address|city|state|zip_code|birth_date|status| age|tenure_years|salary_band|      full_name|email_domain|
+-----------+----------+---------+--------------------+-------------------+--------------------+----------+-------+----------+--------------------+----+-----+--------+----------+------+----+------------+-----------+---------------+------------+
|          0|  Michelle|   Miller|   preed@example.com|2021-07-17 00:00:00|     Patent attorney| analytics|34634.0|      1474|7699 Kenneth Lodg...|NULL| NULL|    NULL|      NULL|  NULL|NULL|         4.7|     Junior|Michelle Miller| example.com|
|          1|    Ama